# BNO085 IMU Sensor Noise Modelling

**Sensors:** Gyroscope · Accelerometer · Magnetometer  
**Dataset:** UNIT_0001_RUN_008 — 54 000 s static capture

This notebook is structured in four sections:

| Section | What it does |
|---|---|
| 0 – Setup | Install deps, mount Drive, load & clean data (reuses your existing pipeline) |
| 1 – Model definition | Define noise model classes for Gyro / Accel / Mag |
| 2 – Parameter fitting | Fit model parameters from real data, confirm against Allan results |
| 3 – Validation | Residual analysis, Q–Q plots, whiteness tests, Kalman-ready parameter tables |

---
## Section 0 — Setup & Data Loading

In [ ]:
# ── 0-A  Dependencies ─────────────────────────────────────────────────────────
!pip install allantools -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats, signal
from scipy.optimize import curve_fit
import allantools
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'lines.linewidth': 0.9,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
COLORS = {'x': '#E74C3C', 'y': '#2ECC71', 'z': '#3498DB', 'mag': '#9B59B6'}
print('✅  Dependencies ready')

In [ ]:
# ── 0-B  Mount Drive & load data (same pipeline as your original notebook) ────
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/capstone_data/UNIT_0001_RUN_008"

def load_sensor(path, sensor_name):
    df = pd.read_csv(path)
    time_col = next(
        (c for c in df.columns if any(k in c.lower() for k in ['time','ts','stamp','ms'])),
        df.columns[0]
    )
    xyz_cols = [c for c in df.columns
                if c.lower() in ['x','y','z']
                or c.lower().endswith(('_x','_y','_z'))]
    if len(xyz_cols) < 3:
        xyz_cols = [c for c in df.columns if c != time_col][:3]
    df = df[[time_col] + xyz_cols].copy()
    df.columns = ['timestamp_ms', 'x', 'y', 'z']
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['time_s'] = (df['timestamp_ms'] - df['timestamp_ms'].iloc[0]) / 1000.0
    df = df.dropna().drop_duplicates().reset_index(drop=True)
    dt_mean = df['timestamp_ms'].diff().dropna().mean()
    fs = round(1000.0 / dt_mean, 2)
    print(f'  [{sensor_name}]  rows={len(df):,}  fs≈{fs} Hz  duration={df.time_s.iloc[-1]:.0f}s')
    return df, fs

df_accel, FS_ACCEL = load_sensor(f"{DATA_ROOT}/accelerometer.csv", "Accel")
df_gyro,  FS_GYRO  = load_sensor(f"{DATA_ROOT}/gyroscope.csv",    "Gyro")
df_mag,   FS_MAG   = load_sensor(f"{DATA_ROOT}/magnetometer.csv", "Mag")
print('\n✅  All sensors loaded')

---
## Section 1 — Noise Model Definitions

### Model equations

**Gyroscope** (ARW-dominant; bias instability τ★ beyond data window)
```
ω_meas[k] = ω_true[k] + b[k] + n[k]
b[k+1]    = b[k] + w_b[k]          (bias random walk)

n[k]   ~ N(0, σ_n²)     σ_n  = ARW / √dt
w_b[k] ~ N(0, σ_b²)     σ_b  = σ_RW · √dt   (conservative estimate)
```

**Accelerometer** (full model: VRW + bias instability both visible in Allan)
```
a_meas[k] = a_true[k] + b[k] + n[k]
b[k+1]    = b[k] + w_b[k]

n[k]   ~ N(0, σ_n²)     σ_n  = VRW / √dt
w_b[k] ~ N(0, σ_b²)     σ_b derived from Allan BI and τ★
```

**Magnetometer** (static hard-iron calibration + per-axis white noise)
```
B_meas[k] = B_true[k] + b_hi + n[k]

b_hi  — constant hard-iron offset, estimated from mean over static window
n[k]  ~ N(0, R_mag)     R_mag = diag(σ_x², σ_y², σ_z²)  from time-domain std
```

In [ ]:
# ── 1  Noise model dataclass ──────────────────────────────────────────────────
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class GyroModel:
    """Gyroscope: ARW white noise + bias random walk (per axis)"""
    # Time-domain stats (rad/s)
    bias:   np.ndarray = field(default_factory=lambda: np.zeros(3))   # [x,y,z]
    sigma_n: np.ndarray = field(default_factory=lambda: np.zeros(3))  # white noise σ (discrete)
    sigma_b: np.ndarray = field(default_factory=lambda: np.zeros(3))  # bias RW σ  (per √s)
    # Allan-derived (continuous)
    ARW:    np.ndarray = field(default_factory=lambda: np.zeros(3))   # rad/s/√Hz
    BI:     Optional[np.ndarray] = None                                # rad/s  (None if not found)
    # Kalman-ready discrete-time covariances (call build_Q after fitting)
    Q_n:  Optional[np.ndarray] = None   # 3×3 measurement noise cov
    Q_b:  Optional[np.ndarray] = None   # 3×3 bias process noise cov

    def build_Q(self, dt: float):
        self.Q_n = np.diag((self.sigma_n)**2)
        self.Q_b = np.diag((self.sigma_b * np.sqrt(dt))**2)


@dataclass
class AccelModel:
    """Accelerometer: VRW white noise + bias random walk (per axis)"""
    bias:    np.ndarray = field(default_factory=lambda: np.zeros(3))
    sigma_n: np.ndarray = field(default_factory=lambda: np.zeros(3))
    sigma_b: np.ndarray = field(default_factory=lambda: np.zeros(3))
    VRW:     np.ndarray = field(default_factory=lambda: np.zeros(3))  # m/s²/√Hz
    BI:      np.ndarray = field(default_factory=lambda: np.zeros(3))  # m/s²
    tau_star: np.ndarray = field(default_factory=lambda: np.zeros(3)) # s
    Q_n:  Optional[np.ndarray] = None
    Q_b:  Optional[np.ndarray] = None

    def build_Q(self, dt: float):
        self.Q_n = np.diag((self.sigma_n)**2)
        self.Q_b = np.diag((self.sigma_b * np.sqrt(dt))**2)


@dataclass
class MagModel:
    """Magnetometer: hard-iron offset + per-axis white noise"""
    hard_iron: np.ndarray = field(default_factory=lambda: np.zeros(3))  # µT
    sigma_n:   np.ndarray = field(default_factory=lambda: np.zeros(3))  # µT
    R_mag:     Optional[np.ndarray] = None   # 3×3 measurement noise cov
    R_mag_scaled: Optional[np.ndarray] = None  # Z-axis upweighted

    def build_R(self, z_scale: float = 20.0):
        self.R_mag = np.diag(self.sigma_n**2)
        s = self.sigma_n.copy()
        s[2] *= z_scale   # Z axis has CV=33.6%, inflate its noise
        self.R_mag_scaled = np.diag(s**2)

print('✅  Model classes defined')

---
## Section 2 — Parameter Fitting

In [ ]:
# ── 2-A  Gyroscope: spike removal → time-domain fit → Allan confirmation ──────

# Step 1: remove synchronous spikes with median filter (kernel=5 ≈ 50 ms at 100 Hz)
for ax in ['x', 'y', 'z']:
    df_gyro[f'{ax}_clean'] = signal.medfilt(df_gyro[ax].values, kernel_size=5)

gyro_model = GyroModel()

# Step 2: bias = mean over full static window (GT = 0)
gyro_model.bias = np.array([
    df_gyro['x_clean'].mean(),
    df_gyro['y_clean'].mean(),
    df_gyro['z_clean'].mean()
])

# Step 3: white noise σ from time-domain std of de-biased signal
dt_gyro = 1.0 / FS_GYRO
for i, ax in enumerate(['x', 'y', 'z']):
    residual = df_gyro[f'{ax}_clean'] - gyro_model.bias[i]
    gyro_model.sigma_n[i] = float(residual.std())

# Step 4: ARW from Allan (τ=1s read-off)
#   From your Allan plot: ARW = 2.19e-2 rad/s/√Hz for all three axes
ARW_VAL = 2.19e-2
gyro_model.ARW = np.array([ARW_VAL, ARW_VAL, ARW_VAL])

# Step 5: conservative bias RW — 10% of time-domain std per √s
#   (will be updated once Allan minimum is found with longer data)
gyro_model.sigma_b = gyro_model.sigma_n * 0.10

# Step 6: build discrete-time covariance matrices
gyro_model.build_Q(dt_gyro)

print('Gyroscope model parameters')
print(f'  Bias    [x,y,z]: {gyro_model.bias}')
print(f'  σ_n     [x,y,z]: {gyro_model.sigma_n}  rad/s')
print(f'  ARW     [x,y,z]: {gyro_model.ARW}  rad/s/√Hz')
print(f'  σ_b     [x,y,z]: {gyro_model.sigma_b}  rad/s/√s  (conservative)')
print(f'  Q_n diagonal   : {np.diag(gyro_model.Q_n)}')
print(f'  Q_b diagonal   : {np.diag(gyro_model.Q_b)}')

In [ ]:
# ── 2-B  Accelerometer: bias calibration → VRW + BI from Allan ────────────────

GRAVITY = 9.81
accel_model = AccelModel()
dt_accel = 1.0 / FS_ACCEL

# Step 1: static bias from mean (X and Y axes have non-zero means due to tilt)
accel_model.bias = np.array([
    df_accel['x'].mean(),          # ≈ +0.456 m/s²
    df_accel['y'].mean(),          # ≈ −0.156 m/s²
    df_accel['z'].mean() - GRAVITY # ≈ −0.051 m/s² (gravity-referenced)
])

# Step 2: calibrate
df_accel['x_cal'] = df_accel['x'] - accel_model.bias[0]
df_accel['y_cal'] = df_accel['y'] - accel_model.bias[1]
df_accel['z_cal'] = df_accel['z'] - accel_model.bias[2] - GRAVITY  # remove gravity

# Step 3: white noise σ from time-domain std of calibrated signal
for i, ax in enumerate(['x_cal', 'y_cal', 'z_cal']):
    accel_model.sigma_n[i] = float(df_accel[ax].std())

# Step 4: VRW from Allan τ=1s read-off
accel_model.VRW = np.array([7.02e-3, 7.49e-3, 9.22e-3])  # m/s²/√Hz

# Step 5: Bias Instability and τ★ from Allan plot minimum
accel_model.BI       = np.array([5.82e-4, 3.56e-4, 1.26e-3])  # m/s²
accel_model.tau_star = np.array([300.0,   800.0,   200.0])    # s

# Step 6: bias RW σ derived from BI and τ★
#   σ_RW = BI × √(2π) / √τ★   (standard Allan-to-RW conversion)
accel_model.sigma_b = (accel_model.BI * np.sqrt(2 * np.pi)
                       / np.sqrt(accel_model.tau_star))

# Step 7: build covariance matrices
accel_model.build_Q(dt_accel)

print('Accelerometer model parameters')
print(f'  Bias    [x,y,z]: {accel_model.bias}  m/s²')
print(f'  σ_n     [x,y,z]: {accel_model.sigma_n}  m/s²')
print(f'  VRW     [x,y,z]: {accel_model.VRW}  m/s²/√Hz')
print(f'  BI      [x,y,z]: {accel_model.BI}  m/s²')
print(f'  τ★      [x,y,z]: {accel_model.tau_star}  s')
print(f'  σ_b     [x,y,z]: {accel_model.sigma_b}  m/s²/√s')
print(f'  Q_n diagonal   : {np.diag(accel_model.Q_n)}')
print(f'  Q_b diagonal   : {np.diag(accel_model.Q_b)}')

In [ ]:
# ── 2-C  Magnetometer: hard-iron calibration + noise cov ─────────────────────

mag_model = MagModel()

# Hard-iron offset = static mean (sensor stationary throughout)
mag_model.hard_iron = np.array([
    df_mag['x'].mean(),   # ≈ +18.14 µT
    df_mag['y'].mean(),   # ≈ −16.93 µT
    df_mag['z'].mean(),   # ≈  +1.95 µT
])

# Calibrated residual
for i, ax in enumerate(['x', 'y', 'z']):
    df_mag[f'{ax}_cal'] = df_mag[ax] - mag_model.hard_iron[i]

# Per-axis noise from time-domain std of calibrated signal
mag_model.sigma_n = np.array([
    df_mag['x_cal'].std(),
    df_mag['y_cal'].std(),
    df_mag['z_cal'].std(),
])

# Build R matrices (normal + Z-axis inflated by factor 20)
mag_model.build_R(z_scale=20.0)

print('Magnetometer model parameters')
print(f'  Hard-iron offset [x,y,z]: {mag_model.hard_iron}  µT')
print(f'  σ_n              [x,y,z]: {mag_model.sigma_n}  µT')
print(f'  R_mag diagonal          : {np.diag(mag_model.R_mag)}')
print(f'  R_mag_scaled diagonal   : {np.diag(mag_model.R_mag_scaled)}')
print(f'  (Z-axis measurement noise inflated ×20 due to CV=33.6%)')

In [ ]:
# ── 2-D  Verify σ_n against Allan ARW: they should agree within ~20% ──────────
#
# Relation: σ_n (discrete, rad/s) = ARW (rad/s/√Hz) / √dt
#           i.e.  ARW_inferred = σ_n × √dt

print('━' * 58)
print('Consistency check: time-domain σ_n  vs  Allan ARW')
print('━' * 58)

print('\nGyroscope:')
arw_inf = gyro_model.sigma_n * np.sqrt(dt_gyro)
for i, ax in enumerate('xyz'):
    ratio = arw_inf[i] / gyro_model.ARW[i]
    flag  = '⚠️ ' if abs(ratio - 1) > 0.3 else '✅'
    print(f'  {ax}: σ_n={gyro_model.sigma_n[i]:.5f}  '
          f'ARW_inferred={arw_inf[i]:.5f}  '
          f'ARW_allan={gyro_model.ARW[i]:.5f}  '
          f'ratio={ratio:.2f}  {flag}')

print('\nAccelerometer:')
vrw_inf = accel_model.sigma_n * np.sqrt(dt_accel)
for i, ax in enumerate('xyz'):
    ratio = vrw_inf[i] / accel_model.VRW[i]
    flag  = '⚠️ ' if abs(ratio - 1) > 0.3 else '✅'
    print(f'  {ax}: σ_n={accel_model.sigma_n[i]:.5f}  '
          f'VRW_inferred={vrw_inf[i]:.5f}  '
          f'VRW_allan={accel_model.VRW[i]:.5f}  '
          f'ratio={ratio:.2f}  {flag}')

print()
print('Note: gyro σ_n may be elevated vs ARW because the time-domain std')
print('still includes any residual low-frequency drift not captured by ARW alone.')

---
## Section 3 — Model Validation

Five tests per sensor:
1. **Residual time series** — visually check for remaining structure (trend, periodicity)
2. **Histogram vs fitted Gaussian** — check for heavy tails / skew
3. **Q–Q plot** — quantify deviation from normality
4. **Autocorrelation (ACF)** — check residual whiteness (should decay to zero within a few lags)
5. **Kolmogorov–Smirnov & Shapiro-Wilk normality tests** — statistical confirmation

In [ ]:
# ── 3-A  Validation helper functions ─────────────────────────────────────────

PLOT_N = 40_000   # downsample for plotting only; stats use full data

def plot_sample(series, n=PLOT_N):
    if len(series) <= n:
        return series
    step = len(series) // n
    return series.iloc[::step].reset_index(drop=True)


def validate_residuals(residual_series, label, color, axes_row, fs_hz,
                       time_s=None, n_lags=200):
    """
    Plot residual timeseries, histogram+Gaussian, Q-Q, and ACF.
    Returns dict of test statistics.
    """
    res = residual_series.dropna().values
    res_plot = plot_sample(pd.Series(res))

    # -- Timeseries --
    ax0 = axes_row[0]
    t_plot = (np.arange(len(res_plot)) / fs_hz) if time_s is None else \
              plot_sample(time_s).values[:len(res_plot)]
    ax0.plot(t_plot, res_plot, color=color, alpha=0.5, lw=0.5)
    ax0.axhline(0, color='k', lw=0.8, ls='--')
    ax0.set_title(f'{label}  residual')
    ax0.set_xlabel('time (s)')

    # -- Histogram vs Gaussian --
    ax1 = axes_row[1]
    mu, sigma = res.mean(), res.std()
    ax1.hist(res, bins=120, density=True, color=color, alpha=0.6, label='data')
    x_fit = np.linspace(mu - 5*sigma, mu + 5*sigma, 400)
    ax1.plot(x_fit, stats.norm.pdf(x_fit, mu, sigma),
             'k-', lw=1.5, label=f'N({mu:.4f}, {sigma:.4f}²)')
    ax1.set_title('Histogram vs Gaussian')
    ax1.legend(fontsize=8)

    # -- Q–Q plot --
    ax2 = axes_row[2]
    sample = np.random.choice(res, size=min(5000, len(res)), replace=False)
    (osm, osr), (slope, intercept, r) = stats.probplot(sample, dist='norm')
    ax2.scatter(osm, osr, s=1, alpha=0.3, color=color)
    ax2.plot(osm, slope*np.array(osm)+intercept, 'k-', lw=1.2)
    ax2.set_title(f'Q–Q  (R²={r**2:.4f})')
    ax2.set_xlabel('theoretical quantiles')
    ax2.set_ylabel('sample quantiles')

    # -- ACF --
    ax3 = axes_row[3]
    sample_acf = np.random.choice(res, size=min(20000, len(res)), replace=False)
    acf_vals = [np.corrcoef(sample_acf[:-k], sample_acf[k:])[0,1]
                for k in range(1, n_lags+1)]
    conf = 1.96 / np.sqrt(len(sample_acf))
    lags = np.arange(1, n_lags+1)
    ax3.bar(lags, acf_vals, color=color, alpha=0.6, width=0.8)
    ax3.axhline( conf, color='k', ls='--', lw=0.8, label='95% conf')
    ax3.axhline(-conf, color='k', ls='--', lw=0.8)
    ax3.axhline(0, color='k', lw=0.5)
    ax3.set_title('ACF of residuals')
    ax3.set_xlabel('lag')
    ax3.legend(fontsize=8)

    # -- Statistical tests (on subsample for speed) --
    sub = np.random.choice(res, size=min(5000, len(res)), replace=False)
    ks_stat, ks_p   = stats.kstest(sub, 'norm', args=(sub.mean(), sub.std()))
    sw_stat, sw_p   = stats.shapiro(sub[:min(5000, len(sub))])
    kurt            = stats.kurtosis(res)
    skew            = stats.skew(res)

    return {'mu': mu, 'sigma': sigma, 'kurtosis': kurt, 'skewness': skew,
            'KS_p': ks_p, 'SW_p': sw_p, 'ACF_lag1': acf_vals[0]}

print('✅  Validation helpers ready')

In [ ]:
# ── 3-B  Gyroscope validation ─────────────────────────────────────────────────
print('Validating Gyroscope model...')

fig, axes_grid = plt.subplots(3, 4, figsize=(18, 11))
fig.suptitle('Gyroscope — residual validation (after spike removal & bias subtraction)',
             fontsize=12, fontweight='bold')

gyro_val_results = {}
for i, ax in enumerate(['x', 'y', 'z']):
    residual = df_gyro[f'{ax}_clean'] - gyro_model.bias[i]
    res = validate_residuals(
        residual, f'Gyro {ax}', COLORS[ax],
        axes_grid[i], FS_GYRO,
        time_s=df_gyro['time_s']
    )
    gyro_val_results[ax] = res

plt.tight_layout()
plt.savefig('gyro_validation.png', bbox_inches='tight')
plt.show()

print('\nGyro residual test summary:')
print(f'{"Axis":>5} {"σ":>10} {"Kurt":>8} {"Skew":>8} {"KS p":>10} {"SW p":>10} {"ACF[1]":>9}')
for ax in 'xyz':
    r = gyro_val_results[ax]
    print(f'{ax:>5} {r["sigma"]:>10.5f} {r["kurtosis"]:>8.3f} '
          f'{r["skewness"]:>8.3f} {r["KS_p"]:>10.4f} {r["SW_p"]:>10.4f} '
          f'{r["ACF_lag1"]:>9.4f}')

print()
print('Interpretation guidance:')
print('  Kurtosis > 3   → heavy tails (spikes still present or non-Gaussian)')
print('  KS / SW p < 0.05 → residuals not Gaussian at 95% confidence')
print('  |ACF[1]| > 0.05  → residuals are NOT white (model missing a component)')

In [ ]:
# ── 3-C  Accelerometer validation ────────────────────────────────────────────
print('Validating Accelerometer model...')

fig, axes_grid = plt.subplots(3, 4, figsize=(18, 11))
fig.suptitle('Accelerometer — residual validation (after bias & gravity subtraction)',
             fontsize=12, fontweight='bold')

accel_val_results = {}
for i, (raw_ax, cal_ax) in enumerate(zip('xyz', ['x_cal','y_cal','z_cal'])):
    residual = df_accel[cal_ax]
    res = validate_residuals(
        residual, f'Accel {raw_ax}', COLORS[raw_ax],
        axes_grid[i], FS_ACCEL,
        time_s=df_accel['time_s']
    )
    accel_val_results[raw_ax] = res

plt.tight_layout()
plt.savefig('accel_validation.png', bbox_inches='tight')
plt.show()

print('\nAccel residual test summary:')
print(f'{"Axis":>5} {"σ":>10} {"Kurt":>8} {"Skew":>8} {"KS p":>10} {"SW p":>10} {"ACF[1]":>9}')
for ax in 'xyz':
    r = accel_val_results[ax]
    print(f'{ax:>5} {r["sigma"]:>10.5f} {r["kurtosis"]:>8.3f} '
          f'{r["skewness"]:>8.3f} {r["KS_p"]:>10.4f} {r["SW_p"]:>10.4f} '
          f'{r["ACF_lag1"]:>9.4f}')

In [ ]:
# ── 3-D  Magnetometer validation ─────────────────────────────────────────────
print('Validating Magnetometer model...')

fig, axes_grid = plt.subplots(3, 4, figsize=(18, 11))
fig.suptitle('Magnetometer — residual validation (after hard-iron subtraction)',
             fontsize=12, fontweight='bold')

mag_val_results = {}
for i, ax in enumerate(['x', 'y', 'z']):
    residual = df_mag[f'{ax}_cal']
    res = validate_residuals(
        residual, f'Mag {ax}', COLORS[ax],
        axes_grid[i], FS_MAG,
        time_s=df_mag['time_s']
    )
    mag_val_results[ax] = res

plt.tight_layout()
plt.savefig('mag_validation.png', bbox_inches='tight')
plt.show()

print('\nMag residual test summary:')
print(f'{"Axis":>5} {"σ":>10} {"Kurt":>8} {"Skew":>8} {"KS p":>10} {"SW p":>10} {"ACF[1]":>9}')
for ax in 'xyz':
    r = mag_val_results[ax]
    print(f'{ax:>5} {r["sigma"]:>10.5f} {r["kurtosis"]:>8.3f} '
          f'{r["skewness"]:>8.3f} {r["KS_p"]:>10.4f} {r["SW_p"]:>10.4f} '
          f'{r["ACF_lag1"]:>9.4f}')

print()
print('Note: Mag Z axis (CV=33.6%) is expected to show non-Gaussian behaviour.')
print('Its measurement noise is inflated ×20 in R_mag_scaled accordingly.')

In [ ]:
# ── 3-E  PSD check: verify -10 dB/decade slope (white noise signature) ────────
#   White noise has a flat power spectral density.
#   If the PSD slopes downward before a rolloff it means a coloured-noise component
#   (e.g. flicker noise, temperature drift) is present.

fig, axes_p = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Power Spectral Density of calibrated residuals',
             fontsize=12, fontweight='bold')

psd_data = [
    ('Gyro',  df_gyro,  ['x_clean','y_clean','z_clean'], FS_GYRO,  [gyro_model.bias[i] for i in range(3)]),
    ('Accel', df_accel, ['x_cal',  'y_cal',  'z_cal'  ], FS_ACCEL, [0,0,0]),
    ('Mag',   df_mag,   ['x_cal',  'y_cal',  'z_cal'  ], FS_MAG,   [0,0,0]),
]

for ax_idx, (name, df, cols, fs, biases) in enumerate(psd_data):
    for j, (col, color) in enumerate(zip(cols, COLORS.values())):
        sig = df[col].dropna().values - biases[j]
        # Welch PSD on downsampled signal for speed
        if len(sig) > 500_000:
            sig = sig[::len(sig)//500_000]
            fs_eff = fs / (len(df[col]) // 500_000)
        else:
            fs_eff = fs
        f, pxx = signal.welch(sig, fs=fs_eff, nperseg=1024)
        axes_p[ax_idx].loglog(f[1:], pxx[1:], color=color,
                              alpha=0.8, lw=0.8, label=col.split('_')[0])
    axes_p[ax_idx].set_title(f'{name} PSD')
    axes_p[ax_idx].set_xlabel('Frequency (Hz)')
    axes_p[ax_idx].set_ylabel('PSD')
    axes_p[ax_idx].legend(fontsize=9)

plt.tight_layout()
plt.savefig('psd_check.png', bbox_inches='tight')
plt.show()

print('Flat PSD → white noise model appropriate.')
print('Rising PSD at low freq → coloured noise / drift → may need higher-order model.')

In [ ]:
# ── 3-F  Final Kalman-ready parameter summary table ───────────────────────────

DT_FUSE = 0.01   # ← set this to your actual Kalman filter fusion period (s)

gyro_model.build_Q(DT_FUSE)
accel_model.build_Q(DT_FUSE)

print('=' * 65)
print(f'  Kalman-ready noise parameters  (dt_fuse = {DT_FUSE} s)')
print('=' * 65)

print('\n--- GYRO  (state: [attitude(3), gyro_bias(3)]) ---')
print(f'  Q_n  (attitude process noise, 3×3 diag):')
print(f'       {np.diag(gyro_model.Q_n)}')
print(f'  Q_b  (bias process noise, 3×3 diag):')
print(f'       {np.diag(gyro_model.Q_b)}')
print(f'  Note: σ_b is conservative. Refit once Allan minimum is found.')

print('\n--- ACCEL  (measurement noise, used in R matrix) ---')
print(f'  R_accel  (3×3 diag, m/s²)²:')
print(f'       {np.diag(accel_model.Q_n)}')
print(f'  Q_b  (accel bias process noise, 3×3 diag):')
print(f'       {np.diag(accel_model.Q_b)}')
print(f'  Note: Z axis Q_b is ~3–7× larger than X/Y — do NOT use a single scalar.')

print('\n--- MAG  (measurement noise, used in R matrix) ---')
print(f'  R_mag  (3×3 diag, µT²)  — standard:')
print(f'       {np.diag(mag_model.R_mag)}')
print(f'  R_mag_scaled  (Z inflated ×20)  — recommended for heading fusion:')
print(f'       {np.diag(mag_model.R_mag_scaled)}')

print('\n--- Copy-paste block (numpy) ---')
print(f'Q_gyro_n    = np.diag({list(np.round(np.diag(gyro_model.Q_n), 8))})')
print(f'Q_gyro_b    = np.diag({list(np.round(np.diag(gyro_model.Q_b), 10))})')
print(f'R_accel     = np.diag({list(np.round(np.diag(accel_model.Q_n), 8))})')
print(f'Q_accel_b   = np.diag({list(np.round(np.diag(accel_model.Q_b), 12))})')
print(f'R_mag       = np.diag({list(np.round(np.diag(mag_model.R_mag), 6))})')
print(f'R_mag_scaled= np.diag({list(np.round(np.diag(mag_model.R_mag_scaled), 4))})')
print('=' * 65)

---
## What to do next based on validation results

| Observation | Diagnosis | Action |
|---|---|---|
| Gyro kurtosis >> 3 even after medfilt | Spikes not fully removed | Increase `kernel_size` to 7 or 9; or use IQR-based clipping |
| Gyro ACF[1] >> 0.05 | Residual is coloured — bias drifts slowly | Add a Gauss-Markov bias state with time constant τ★ (once Allan minimum found) |
| Accel Z ACF shows periodicity | Periodic environmental vibration | Apply a notch filter at the offending frequency before modelling |
| Mag Z non-Gaussian / high kurtosis | Expected (CV=33.6%) | Keep Z-axis R inflated; consider excluding from heading fusion entirely |
| Any PSD rising at low freq | Drift not captured by random-walk model | Add a first-order Gauss-Markov process with correlation time = 1/f_knee |

**When to re-run Allan for gyro BI:**  
Collect a new static dataset of at least 6–8 hours under stable temperature, then re-run the Allan cell from your original notebook. The minimum should appear around τ ≈ 10⁴–10⁵ s based on the current downward trend.